# Reverse-engineering production/decay features

Goal: determine whether the provided dynamic features can be reproduced from `patient.xlsx` / supervisor workbook raw visit values and timing variables.

This is intentionally an audit notebook. It should not assume the formulas. It first verifies raw serum IgG visit columns against the 63-feature matrix, then tests candidate production/decay formulas. A formula should only be promoted into the paper pipeline when it matches all 91 rows to numerical tolerance.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CWD = Path.cwd().resolve()
if (CWD / 'data_synthesis').exists():
    REPO_ROOT = CWD
    DS_ROOT = REPO_ROOT / 'data_synthesis'
elif CWD.name == 'notebooks':
    DS_ROOT = CWD.parent
    REPO_ROOT = DS_ROOT.parent
else:
    DS_ROOT = CWD
    REPO_ROOT = DS_ROOT.parent
OUTDIR = DS_ROOT / 'notebooks' / 'virtual_patient_outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)

feature_path = DS_ROOT / 'data' / 'features_withMissForestImputation_IR_INR_together_jul25_63Features.csv'
patient_path = DS_ROOT / 'data' / 'patient.xlsx'
supervisor_patient_path = REPO_ROOT / 'randomForest_HIV_vaccine-main' / 'humanData' / 'Data-2_withMemoryBForsomeHIVpositive.xlsx'

feature = pd.read_csv(feature_path).rename(columns={'Unnamed: 0': 'participant_index'})
patient = pd.read_excel(patient_path)
supervisor_patient = pd.read_excel(supervisor_patient_path)

print(feature.shape, patient.shape, supervisor_patient.shape)


(91, 64) (1311, 54) (1311, 54)


## 1. Build participant-level visit table from the workbook

The workbook uses participant-start rows followed by visit rows. We forward-fill participant index, ID, HIV status, and dose/timing metadata into the visit rows. This preserves the visit-level raw data without duplicating the 63-feature matrix.

In [2]:
def coerce_numeric_series(s):
    return pd.to_numeric(s.replace({'TND<40': 0, '<40': 0, '< 50.06': 50.06}), errors='coerce')

VISIT_NORMALIZATION = {
    'V8c/V9': 'V9',
    'V8c': 'V9',
    'V8/8a': 'V8a',
    'V8/V8a': 'V8a',
    'V8-1': 'V8',
    'V4/V4a': 'V4a',
}


def make_visit_table(raw):
    df = raw.copy()
    id_cols = ['#', "Participant's ID", 'Age', 'HIV status: HIV Neg = 0, IR=1, INR=2, LLV=3, LTNP=4',
               'Days between D1 and D2', 'Days between D2 and V8', 'Days between D3 and V9']
    for c in id_cols:
        if c in df.columns:
            df[c] = df[c].ffill()
    df = df.rename(columns={
        '#': 'participant_index',
        "Participant's ID": 'participant_id',
        'HIV status: HIV Neg = 0, IR=1, INR=2, LLV=3, LTNP=4': 'hiv_status',
        'Days after D1 (Dose 1)': 'days_after_D1',
        'Days between D1 and D2': 'days_D1_D2',
        'Days between D2 and V8': 'days_D2_V8',
        'Days between D3 and V9': 'days_D3_V9',
        'Serum - IgG levels': 'serum_IgG_NP',
        'Unnamed: 30': 'serum_IgG_RBD',
        'Unnamed: 31': 'serum_IgG_spike',
        'ACE2 snELISA                        Neutralisation, IU/mL': 'ACE2_neutralization',
        'CD4/CD8': 'CD4_CD8',
    })
    df['visit_raw'] = df['Visit #'].astype(str)
    df['visit'] = df['visit_raw'].replace(VISIT_NORMALIZATION)
    df['participant_index'] = pd.to_numeric(df['participant_index'], errors='coerce')
    df = df[df['participant_index'].notna()].copy()
    df['participant_index'] = df['participant_index'].astype(int)
    for c in ['days_after_D1', 'days_D1_D2', 'days_D2_V8', 'days_D3_V9', 'serum_IgG_RBD', 'serum_IgG_spike', 'CD4_CD8']:
        if c in df.columns:
            df[c] = coerce_numeric_series(df[c])
    return df

visits = make_visit_table(patient)
sup_visits = make_visit_table(supervisor_patient)

display(visits[['participant_index', 'participant_id', 'visit_raw', 'visit', 'days_after_D1', 'serum_IgG_spike', 'serum_IgG_RBD']].head(20))
visit_counts = visits['visit'].value_counts().sort_index()
display(visit_counts)


,participant_index,participant_id,visit_raw,visit,days_after_D1,serum_IgG_spike,serum_IgG_RBD
1,1,OM5402,Screening,Screening,NaN,NaN,NaN
2,1,OM5402,V1,V1,NaN,46.40,38.33
3,1,OM5402,V2,V2,10.0,NaN,NaN
4,1,OM5402,V3,V3,20.0,NaN,NaN
5,1,OM5402,V4,V4,27.0,731.08,759.68
6,1,OM5402,V4a,V4a,56.0,436.85,420.48
7,1,OM5402,V5,V5,62.0,NaN,NaN
8,1,OM5402,V6,V6,69.0,412.08,444.06
9,1,OM5402,V7,V7,83.0,NaN,NaN
10,1,OM5402,V8,V8,165.0,103.63,167.00


visit
Screening     91
V1            91
V2            91
V3            91
V4            90
V4a           91
V5            91
V6            91
V7            91
V8            83
V8a           91
V8b           91
V9           137
Name: count, dtype: int64

## 2. Verify raw serum IgG visit values against the 63-feature matrix

This checks whether columns such as `V4_blood_IgGspike` are directly copied from the workbook. If this fails, exact formula reconstruction will require another source file.

In [3]:
def pivot_serum(visits, value_col):
    return visits.pivot_table(index='participant_index', columns='visit', values=value_col, aggfunc='first')

spike_piv = pivot_serum(visits, 'serum_IgG_spike')
rbd_piv = pivot_serum(visits, 'serum_IgG_RBD')

checks = []
for antigen, piv, suffix in [('spike', spike_piv, 'blood_IgGspike'), ('RBD', rbd_piv, 'blood_IgGRBD')]:
    for c in feature.columns:
        if not c.endswith(suffix):
            continue
        visit = c.split('_')[0]
        if visit not in piv.columns:
            checks.append({'feature': c, 'visit_in_patient': False, 'n_compare': 0, 'max_abs_diff': np.nan, 'mean_abs_diff': np.nan})
            continue
        merged = feature[['participant_index', c]].merge(piv[[visit]].reset_index(), on='participant_index', how='left')
        valid = merged[c].notna() & merged[visit].notna()
        diff = (merged.loc[valid, c] - merged.loc[valid, visit]).abs()
        checks.append({
            'feature': c,
            'visit_in_patient': True,
            'n_compare': int(valid.sum()),
            'max_abs_diff': float(diff.max()) if len(diff) else np.nan,
            'mean_abs_diff': float(diff.mean()) if len(diff) else np.nan,
        })
raw_match = pd.DataFrame(checks)
display(raw_match)


,feature,visit_in_patient,n_compare,max_abs_diff,mean_abs_diff
0,V1_blood_IgGspike,True,56,735.2414,13.129311
1,V4_blood_IgGspike,True,54,206.7566,3.828826
2,V4a_blood_IgGspike,True,46,63.7268,1.385365
3,V6_blood_IgGspike,True,38,0.0000,0.000000
4,V8_blood_IgGspike,True,77,904.4909,11.746635
5,V8a_blood_IgGspike,True,56,433.1446,23.153079
6,V8b_blood_IgGspike,True,25,0.0000,0.000000
7,V9_blood_IgGspike,True,54,0.0000,0.000000
8,V10_blood_IgGspike,False,0,NaN,NaN
9,V11_blood_IgGspike,False,0,NaN,NaN


## 3. Candidate formulas for production/decay

These are hypotheses, not accepted formulas. They test common longitudinal summaries:

- log10 peak level after dose window;
- log10 fold-change from baseline to peak;
- exponential decay slope on log10 scale per day;
- post-D3 boost from V8/V8a to V8b/V9;
- trapezoidal AUC on raw or log scale.

A match should have near-zero max absolute error after rounding, not merely high correlation.

In [4]:
def safe_log10(x):
    x = np.asarray(x, dtype=float)
    return np.log10(np.where(x > 0, x, np.nan))

# Use feature matrix visit values for candidate reconstruction first, because these are the exact imputed values used downstream.
visit_order = ['V1', 'V4', 'V4a', 'V6', 'V8', 'V8a', 'V8b', 'V9', 'V10', 'V11']
# Approximate visit days from patient workbook medians; V10/V11 are absent from the workbook and remain unknown.
day_medians = visits.groupby('visit')['days_after_D1'].median().to_dict()
day_medians


{'Screening': nan,
 'V1': nan,
 'V2': 10.0,
 'V3': 20.0,
 'V4': 28.0,
 'V4a': 58.0,
 'V5': 67.0,
 'V6': 76.0,
 'V7': 86.0,
 'V8': 167.5,
 'V8a': 208.0,
 'V8b': 237.5,
 'V9': 334.5}

In [5]:
def feature_trajectory(row, suffix, visits_subset):
    vals = []
    for v in visits_subset:
        col = f'{v}_{suffix}'
        vals.append(row[col] if col in row.index else np.nan)
    return np.asarray(vals, dtype=float)

D1D2_VISITS = ['V1', 'V4', 'V4a', 'V6', 'V8', 'V8a']
D3_VISITS = ['V8', 'V8a', 'V8b', 'V9']
DAYS_D1D2 = np.asarray([day_medians.get(v, np.nan) for v in D1D2_VISITS], dtype=float)
DAYS_D3 = np.asarray([day_medians.get(v, np.nan) for v in D3_VISITS], dtype=float)

print('D1D2 visit days used for candidate tests:', dict(zip(D1D2_VISITS, DAYS_D1D2)))
print('D3 visit days used for candidate tests:', dict(zip(D3_VISITS, DAYS_D3)))


def candidate_features(row, suffix):
    d = {}
    y12 = feature_trajectory(row, suffix, D1D2_VISITS)
    log12 = safe_log10(y12)
    # production candidates before/around first two-dose response
    d['log10_peak_D1D2'] = np.nanmax(log12)
    d['log10_fold_V1_to_peak_D1D2'] = np.nanmax(log12) - log12[0]
    d['raw_peak_D1D2'] = np.nanmax(y12)
    # decay candidates after D1D2: peak to V8a, or linear slope after peak
    peak_i = int(np.nanargmax(log12)) if np.isfinite(log12).any() else 0
    if np.isfinite(log12[peak_i]) and np.isfinite(log12[-1]) and np.isfinite(DAYS_D1D2[peak_i]) and np.isfinite(DAYS_D1D2[-1]) and DAYS_D1D2[-1] != DAYS_D1D2[peak_i]:
        d['decay_log10_peak_to_V8a_per_day'] = (log12[peak_i] - log12[-1]) / (DAYS_D1D2[-1] - DAYS_D1D2[peak_i])
    else:
        d['decay_log10_peak_to_V8a_per_day'] = np.nan
    valid = np.isfinite(log12) & np.isfinite(DAYS_D1D2) & (DAYS_D1D2 >= DAYS_D1D2[peak_i])
    if valid.sum() >= 2:
        slope = np.polyfit(DAYS_D1D2[valid], log12[valid], 1)[0]
        d['decay_log10_lm_after_peak_D1D2'] = -slope
    else:
        d['decay_log10_lm_after_peak_D1D2'] = np.nan

    y3 = feature_trajectory(row, suffix, D3_VISITS)
    log3 = safe_log10(y3)
    d['log10_peak_D3'] = np.nanmax(log3)
    d['log10_fold_preD3_to_peak_D3'] = np.nanmax(log3) - np.nanmin(log3[:2])
    d['log10_fold_V8_to_V8b'] = log3[2] - log3[0] if np.isfinite(log3[2]) and np.isfinite(log3[0]) else np.nan
    peak3_i = int(np.nanargmax(log3)) if np.isfinite(log3).any() else 0
    if np.isfinite(log3[peak3_i]) and np.isfinite(log3[-1]) and np.isfinite(DAYS_D3[peak3_i]) and np.isfinite(DAYS_D3[-1]) and DAYS_D3[-1] != DAYS_D3[peak3_i]:
        d['decay_log10_peak_to_V9_per_day'] = (log3[peak3_i] - log3[-1]) / (DAYS_D3[-1] - DAYS_D3[peak3_i])
    else:
        d['decay_log10_peak_to_V9_per_day'] = np.nan
    return d

candidate_rows = []
for _, row in feature.iterrows():
    base = {'participant_index': int(row['participant_index'])}
    for antigen, suffix in [('spike', 'blood_IgGspike'), ('RBD', 'blood_IgGRBD')]:
        for name, value in candidate_features(row, suffix).items():
            base[f'{antigen}__{name}'] = value
    candidate_rows.append(base)
candidates = pd.DataFrame(candidate_rows)
display(candidates.head())


D1D2 visit days used for candidate tests: {'V1': np.float64(nan), 'V4': np.float64(28.0), 'V4a': np.float64(58.0), 'V6': np.float64(76.0), 'V8': np.float64(167.5), 'V8a': np.float64(208.0)}
D3 visit days used for candidate tests: {'V8': np.float64(167.5), 'V8a': np.float64(208.0), 'V8b': np.float64(237.5), 'V9': np.float64(334.5)}


,participant_index,spike__log10_peak_D1D2,spike__log10_fold_V1_to_peak_D1D2,spike__raw_peak_D1D2,spike__decay_log10_peak_to_V8a_per_day,spike__decay_log10_lm_after_peak_D1D2,spike__log10_peak_D3,spike__log10_fold_preD3_to_peak_D3,spike__log10_fold_V8_to_V8b,spike__decay_log10_peak_to_V9_per_day,RBD__log10_peak_D1D2,RBD__log10_fold_V1_to_peak_D1D2,RBD__raw_peak_D1D2,RBD__decay_log10_peak_to_V8a_per_day,RBD__decay_log10_lm_after_peak_D1D2,RBD__log10_peak_D3,RBD__log10_fold_preD3_to_peak_D3,RBD__log10_fold_V8_to_V8b,RBD__decay_log10_peak_to_V9_per_day
0,1,2.863965,1.197447,731.08,0.005201,0.005373,3.109899,1.182067,1.094414,0.004591,2.880631,1.297092,759.68,0.004285,0.004131,3.124977,1.015567,0.902260,0.007774
1,2,2.990490,2.829122,978.34,0.004410,0.004191,3.117960,0.709568,0.413029,0.001947,3.069509,2.576749,1173.57,0.003847,0.003669,3.225482,0.663785,0.412456,0.003533
2,3,3.176323,2.921050,1500.80,0.005784,0.005445,3.176323,0.763526,0.346485,0.002390,3.602091,3.122085,4000.29,0.008974,0.008780,3.543790,1.126235,0.658785,0.007848
3,4,3.176323,2.229871,1500.80,0.002578,0.002529,3.176323,0.340309,0.200641,NaN,3.586488,3.093728,3859.12,0.006776,0.006907,3.648749,0.956696,0.401481,NaN
4,5,2.735224,1.472061,543.53,0.005093,0.005117,3.176323,1.113403,0.919966,0.001868,2.685894,2.205887,485.17,0.005842,0.005955,3.644766,1.730000,1.554226,0.007358


In [6]:
targets = [
    'spikeProduction_D1D2', 'spikeDecay_D1D2', 'spikeProduction_D3', 'spikeDecay_D3',
    'RBDProduction_D1D2', 'RBDDecay_D1D2', 'RBDProduction_D3', 'RBDDecay_D3',
]
wide = feature[['participant_index'] + targets].merge(candidates, on='participant_index', how='left')

scores = []
for target in targets:
    antigen = 'spike' if target.startswith('spike') else 'RBD'
    cand_cols = [c for c in candidates.columns if c.startswith(f'{antigen}__')]
    for cand in cand_cols:
        valid = wide[target].notna() & wide[cand].notna()
        if valid.sum() < 10:
            continue
        x = wide.loc[valid, cand].to_numpy(float)
        t = wide.loc[valid, target].to_numpy(float)
        corr = np.corrcoef(x, t)[0, 1]
        # Best affine calibration. Exact formulas should not need this, but it helps identify shape matches.
        A = np.vstack([x, np.ones_like(x)]).T
        beta, intercept = np.linalg.lstsq(A, t, rcond=None)[0]
        pred = beta * x + intercept
        scores.append({
            'target': target,
            'candidate': cand,
            'n': int(valid.sum()),
            'corr': float(corr),
            'affine_beta': float(beta),
            'affine_intercept': float(intercept),
            'affine_rmse': float(np.sqrt(np.mean((pred - t) ** 2))),
            'affine_max_abs_error': float(np.max(np.abs(pred - t))),
            'raw_max_abs_error': float(np.max(np.abs(x - t))),
        })
score_df = pd.DataFrame(scores).sort_values(['target', 'affine_rmse'])
display(score_df.groupby('target').head(5))


,target,candidate,n,corr,affine_beta,affine_intercept,affine_rmse,affine_max_abs_error,raw_max_abs_error
51,RBDDecay_D1D2,RBD__log10_fold_preD3_to_peak_D3,91,0.419190,1.994106e-04,0.021328,0.000217,0.000757,2.641869
45,RBDDecay_D1D2,RBD__log10_peak_D1D2,91,-0.370794,-1.707293e-04,0.022080,0.000222,0.000798,3.627724
49,RBDDecay_D1D2,RBD__decay_log10_lm_after_peak_D1D2,89,0.154215,1.523133e-02,0.021466,0.000224,0.000746,0.021356
48,RBDDecay_D1D2,RBD__decay_log10_peak_to_V8a_per_day,89,0.134954,1.388630e-02,0.021476,0.000224,0.000729,0.021316
50,RBDDecay_D1D2,RBD__log10_peak_D3,91,-0.333562,-2.736145e-04,0.022486,0.000226,0.000771,3.627724
68,RBDDecay_D3,RBD__log10_peak_D3,91,-0.452744,-3.468242e-04,0.012552,0.000199,0.000471,3.637836
67,RBDDecay_D3,RBD__decay_log10_lm_after_peak_D1D2,89,0.169431,1.581205e-02,0.011285,0.000211,0.000666,0.011344
66,RBDDecay_D3,RBD__decay_log10_peak_to_V8a_per_day,89,0.168223,1.635573e-02,0.011285,0.000211,0.000664,0.011327
64,RBDDecay_D3,RBD__log10_fold_V1_to_peak_D1D2,91,-0.286988,-1.202101e-04,0.011671,0.000214,0.000606,3.157829
65,RBDDecay_D3,RBD__raw_peak_D1D2,91,-0.243629,-3.845655e-08,0.011447,0.000217,0.000605,4453.979086


## 4. Interpretation rules

- If a candidate has high correlation but non-trivial absolute error, it is **not** an exact formula.
- If raw workbook values reproduce visit columns but no candidate reproduces production/decay, the dynamic features likely came from an external model-fitting script that is not in the repository.
- If production/decay features dominate RF distinguishability, the paper can safely claim synthetic generators fail to preserve **provided longitudinal antibody-dynamic summaries**.
- Do not call these mechanistic production/decay rates until the formula or model-fitting procedure is recovered.